In [0]:
"""
Eurostat GDP Volume Loader Notebook
-----------------------------------

This notebook handles the extraction of GDP volume indices from the Eurostat SDMX 2.1 API. It uses the `eurostat` Python package to fetch economic datasets, transform them into Spark DataFrames, and load them into the processed layer of the Unity Catalog.
"""

In [0]:
"""
Install the Eurostat PyPI package into the notebook kernel. This package provides
access to the Eurostat SDMX 2.1 API, enabling the retrieval of economic and
statistical datasets used in ETL pipelines. The installation is scoped to the
notebook environment and does not modify the underlying cluster image. Once
installed, the module can be imported and used for data ingestion tasks.
"""

%pip install eurostat

In [0]:
# Import required libraries
import eurostat
import pandas as pd
from pyspark.sql import functions as F

# Download del dataset Eurostat: 'nama_10_gdp'
# Questo dataset contiene il PIL e le principali componenti economiche per i paesi europei
print("Downloading Eurostat nama_10_gdp dataset...")
# Utilizziamo il metodo 'get_data_df' del pacchetto eurostat per scaricare e creare il DataFrame in un unico passaggio.
df_eurostat = eurostat.get_data_df('nama_10_gdp')
print(f"Dataset downloaded: {df_eurostat.shape[0]} rows, {df_eurostat.columns.tolist()}")

# Stampiamo il risultato per confermare quante righe e colonne sono state caricate
df_eurostat.head(10)

"""
Descrizione dei campi del dataset Eurostat 'nama_10_gdp'

FREQ:
    Indica la frequenza della serie temporale. Nel caso del PIL europeo è 'A',
    cioè valori annuali. Serve a capire ogni quanto viene misurata la variabile.

UNIT:
    Specifica l'unità di misura e il tipo di prezzo utilizzato.
    Esempio: CLV05_MEUR significa:
        - MEUR: milioni di euro
        - CLV05: valori concatenati con anno base 2005
    I valori concatenati (chain-linked volumes) sono serie in termini reali:
    Eurostat elimina l'effetto dell'inflazione ricostruendo ogni anno come se
    i prezzi fossero sempre quelli del 2005. Questo permette confronti coerenti
    nel tempo, perché la crescita riflette variazioni economiche reali e non
    cambiamenti nei prezzi.

NA_ITEM:
    Identifica la variabile economica misurata.
    Esempio: B1G = Prodotto Interno Lordo (PIL) a prezzi di mercato.
    Questo campo dice "che cosa" rappresentano i valori numerici.

GEO:
    Codice del paese secondo la nomenclatura Eurostat.
    Esempi: AL = Albania, AT = Austria, BE = Belgio, BG = Bulgaria, CH = Svizzera.
    Indica "dove" si riferisce la serie.

TIME_PERIOD:
    L'anno di riferimento per ciascun valore della serie.
    Ogni colonna numerica dopo i metadati rappresenta il PIL reale del paese
    in un determinato anno, espresso in milioni di euro a prezzi concatenati.

Sintesi:
    - NA_ITEM = cosa viene misurato (es. PIL)
    - UNIT = come è misurato (milioni di euro, prezzi concatenati 2005)
    - GEO = dove (paese)
    - TIME_PERIOD = quando (anno)
    - I valori concatenati garantiscono confronti temporali privi di inflazione.
"""


In [0]:
# Filter for:
# - Indicator B1GQ: Gross domestic product at market prices
# - Unit CLV15_MEUR: Chain linked volumes (2015=100), million euro
# 
# IMPORTANT: Chain linked volumes eliminate exchange rate and inflation distortions.
# This shows REAL economic growth, making cross-country comparisons accurate.
# Unlike nominal GDP in USD (affected by EUR/USD movements), this measures actual
# economic output in constant prices.

print("\nFiltering for GDP volume indicator...")
df_filtered = df_eurostat[
    (df_eurostat['na_item'] == 'B1GQ') &  # GDP at market prices
    (df_eurostat['unit'] == 'CLV15_MEUR')  # Chain linked volumes, 2015 reference year
]

print(f"After indicator filter: {df_filtered.shape[0]} rows")
df_filtered.head()

In [0]:
# Filter for major European economies
countries = ['IT', 'FR', 'DE', 'ES', 'UK', 'NL']  # Italy, France, Germany, Spain, UK, Netherlands

# Eurostat uses 'geo\\TIME_PERIOD' as column name
geo_col = 'geo\\TIME_PERIOD'

print("\nFiltering for countries:", countries)
df_countries = df_filtered[df_filtered[geo_col].isin(countries)]

print(f"After country filter: {df_countries.shape[0]} rows")

# The Eurostat data has years as columns (wide format)
# We need to identify year columns and filter for 1999-2024
year_columns = [col for col in df_countries.columns if col.isdigit() and 1999 <= int(col) <= 2024]
print(f"\nYear columns found: {len(year_columns)} columns from {min(year_columns)} to {max(year_columns)}")

# Keep only relevant columns: geo + year columns
keep_cols = [geo_col] + year_columns
df_years = df_countries[keep_cols]

print(f"\nFinal filtered dataset: {df_years.shape[0]} rows x {df_years.shape[1]} columns")
df_years

In [0]:
# Transform from wide format (years as columns) to long format (year as row)
# This makes it easier to work with in Spark and for time series analysis

geo_col = 'geo\\TIME_PERIOD'

print("\nTransforming to long format...")
df_long = df_years.melt(
    id_vars=[geo_col],
    value_vars=year_columns,
    var_name='year',
    value_name='gdp_volume_meur'
)

# Convert year to integer
df_long['year'] = df_long['year'].astype(int)

# Add country names
country_names = {
    'IT': 'Italy',
    'FR': 'France',
    'DE': 'Germany',
    'ES': 'Spain',
    'UK': 'United Kingdom',
    'NL': 'Netherlands'
}
df_long['country_name'] = df_long[geo_col].map(country_names)

# Rename geo column to country_code
df_long = df_long.rename(columns={geo_col: 'country_code'})

# Remove rows with missing GDP values
df_long = df_long.dropna(subset=['gdp_volume_meur'])

# Reorder columns
df_long = df_long[['country_code', 'country_name', 'year', 'gdp_volume_meur']]

# Sort by year and country
df_long = df_long.sort_values(['year', 'country_code']).reset_index(drop=True)

print(f"\nTransformed dataset: {df_long.shape[0]} rows")
print(f"Years covered: {df_long['year'].min()} to {df_long['year'].max()}")
print(f"Countries: {df_long['country_name'].unique().tolist()}")

df_long.head(20)

In [0]:
# Convert pandas DataFrame to Spark DataFrame
print("\nConverting to Spark DataFrame...")
spark_df = spark.createDataFrame(df_long)

# Display schema
print("\nSchema:")
spark_df.printSchema()

# Save to Unity Catalog table
table_name = "geopolitics_data_catalog.processed.eurostat_gdp_volume"

print(f"\nSaving to Unity Catalog table: {table_name}")
spark_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(table_name)

print(f"✅ Data successfully saved to {table_name}")

# Verify the saved table
print(f"\nVerifying saved table...")
verify_df = spark.table(table_name)
print(f"Total rows: {verify_df.count()}")
print(f"\nSample data:")
display(verify_df.orderBy("year", "country_code").limit(20))

In [0]:
# Show GDP volume index for 1999 and 2008 to verify Italy's stagnation
print("\n" + "="*60)
print("GDP VOLUME INDEX COMPARISON: 1999 vs 2008")
print("Chain linked volumes (2015=100) - Shows REAL economic growth")
print("="*60)

verify_df = spark.table("geopolitics_data_catalog.processed.eurostat_gdp_volume")

# Get 1999 baseline
df_1999 = verify_df.filter(F.col("year") == 1999) \
    .select("country_name", F.col("gdp_volume_meur").alias("gdp_1999"))

# Get 2008 values
df_2008 = verify_df.filter(F.col("year") == 2008) \
    .select("country_name", F.col("gdp_volume_meur").alias("gdp_2008"))

# Calculate growth
df_comparison = df_1999.join(df_2008, "country_name")
df_comparison = df_comparison.withColumn(
    "growth_pct",
    F.round(100 * (F.col("gdp_2008") - F.col("gdp_1999")) / F.col("gdp_1999"), 1)
).withColumn(
    "index_2008",
    F.round(100 * F.col("gdp_2008") / F.col("gdp_1999"), 1)
)

print("\n📊 Real GDP growth 1999-2008 (chain linked volumes):")
print("   This eliminates exchange rate and inflation distortions\n")

display(df_comparison.orderBy(F.col("growth_pct").desc()))

print("\n💡 Key insight: Italy's GDP grew only ~10-15% in real terms (1999-2008)")
print("   while Germany/France/UK grew 15-25%. The stagnation is now visible!")